In [ ]:
# Install required packages (auto-skipped if already installed)
import importlib
if importlib.util.find_spec('qiskit') is None:
    !pip install -q qiskit qiskit-aer qiskit-ibm-runtime pylatexenc
else:
    print("\u2713 Packages already installed")

# To run on real quantum hardware, uncomment and fill in your credentials:
# from qiskit_ibm_runtime import QiskitRuntimeService
# QiskitRuntimeService.save_account(
#     token="<your-api-key>",
#     instance="<your-crn>",
#     overwrite=True
# )

<details>
<summary><b>Versões dos pacotes</b></summary>

O código nesta página foi desenvolvido usando os seguintes requisitos.
Recomendamos usar essas versões ou mais recentes.

```
qiskit[all]~=2.3.0
```
</details>
Se você tem um conjunto de $n$ bits (ou qubits), normalmente vai rotular cada bit de $0
\rightarrow n-1$. Diferentes softwares e recursos precisam escolher como ordenar
esses bits tanto na memória do computador quanto quando exibidos na tela.

## Convenções do Qiskit

Veja como o Qiskit SDK ordena os bits em diferentes cenários.

### Circuits quânticos

A classe `QuantumCircuit` armazena seus qubits em uma lista
(`QuantumCircuit.qubits`). O índice de um qubit nessa lista define o
rótulo do qubit.

In [1]:
from qiskit import QuantumCircuit, QuantumRegister
from qiskit.circuit import Qubit

qc = QuantumCircuit(2)
qc.qubits[0]  # qubit "0"

Qubit(QuantumRegister(2, "q"), 0)

<Qubit register=(2, "q"), index=0>

### Circuit diagrams

On a circuit diagram, qubit $0$ is the topmost qubit, and qubit $n-1$ the
bottommost qubit. You can change this with the `reverse_bits` argument of
`QuantumCircuit.draw` (see [Change ordering in
Qiskit](#change-ordering-in-qiskit)).

In [2]:
qc.x(1)
qc.draw()

          
q_0: ─────
     ┌───┐
q_1: ┤ X ├
     └───┘

### Diagramas de circuito
Em um diagrama de circuito, o qubit $0$ é o qubit mais ao topo e o qubit $n-1$ é o
mais abaixo. Você pode alterar isso com o argumento `reverse_bits` de
`QuantumCircuit.draw` (veja [Alterar a ordenação no
Qiskit](#change-ordering-in-qiskit)).

In [3]:
from qiskit.primitives import StatevectorSampler as Sampler

qc.measure_all()

job = Sampler().run([qc])
result = job.result()
print(f" > Counts: {result[0].data.meas.get_counts()}")

 > Counts: {'10': 1024}


### Strings

When displaying or interpreting a list of bits (or qubits) as a string, bit
$n-1$ is the leftmost bit, and bit $0$ is the rightmost bit. This is because we
usually write numbers with the most significant digit on the left, and in
Qiskit, bit $n-1$ is interpreted as the most significant bit.

For example, the following cell defines a `Statevector` from a string of
single-qubit states. In this case, qubit $0$ is in state $|+\rangle$, and
qubit $1$ in state $|0\rangle$.

In [4]:
from qiskit.quantum_info import Statevector

sv = Statevector.from_label("0+")
sv.probabilities_dict()

{np.str_('00'): np.float64(0.4999999999999999),
 np.str_('01'): np.float64(0.4999999999999999)}

### Inteiros
Ao interpretar bits como um número, o bit $0$ é o bit menos significativo e o
bit $n-1$ é o mais significativo. Isso é útil na programação porque cada bit tem
o valor $2^\text{label}$ (label sendo o índice do qubit em
`QuantumCircuit.qubits`). Por exemplo, a execução do circuito a seguir termina
com o bit $0$ sendo `0` e o bit $1$ sendo `1`. Isso é interpretado como o
inteiro decimal `2` (medido com probabilidade `1.0`).

In [5]:
print(sv[1])  # amplitude of state |01>
print(sv[2])  # amplitude of state |10>

(0.7071067811865475+0j)
0j


### Gates

Each gate in Qiskit can interpret a list of qubits in its own way, but
controlled gates usually follow the convention `(control, target)`.

For example, the following cell adds a controlled-X gate where qubit $0$ is
the control and qubit $1$ is the target.

In [6]:
from qiskit import QuantumCircuit

qc = QuantumCircuit(2)
qc.cx(0, 1)
qc.draw()

          
q_0: ──■──
     ┌─┴─┐
q_1: ┤ X ├
     └───┘

### Strings
Ao exibir ou interpretar uma lista de bits (ou qubits) como uma string, o bit
$n-1$ é o bit mais à esquerda e o bit $0$ é o mais à direita. Isso ocorre porque
geralmente escrevemos números com o dígito mais significativo à esquerda e, no
Qiskit, o bit $n-1$ é interpretado como o bit mais significativo.

Por exemplo, a célula a seguir define um `Statevector` a partir de uma string de
estados de qubit único. Nesse caso, o qubit $0$ está no estado $|+\rangle$ e o
qubit $1$ no estado $|0\rangle$.

In [7]:
from qiskit import QuantumCircuit

qc = QuantumCircuit(2)
qc.x(0)
qc.draw(reverse_bits=True)

          
q_1: ─────
     ┌───┐
q_0: ┤ X ├
     └───┘

You can use the `reverse_bits` method to return a new circuit with the
qubits' labels reversed (this does not mutate the original circuit).

In [8]:
qc.reverse_bits().draw()

          
q_0: ─────
     ┌───┐
q_1: ┤ X ├
     └───┘

Isso às vezes gera confusão ao interpretar uma string de bits, pois você pode
esperar que o bit mais à esquerda seja o bit $0$, enquanto ele geralmente representa o bit
$n-1$.

### Matrizes de vetor de estado
Ao representar um vetor de estado como uma lista de números complexos (amplitudes),
o Qiskit ordena essas amplitudes de modo que a amplitude no índice $x$ represente
o estado da base computacional $|x\rangle$.